# Automatic Differentiation visualised with Mermaid

This notebook demonstrates how to automatically generate visually distinct Mermaid diagrams for arbitrary computational graphs and their derivatives using `SymPy`.

## Understanding the Computation Graph

The graph shows how an expression is evaluated in the forward pass and how its gradients are calculated in the backward pass. Read the forward computation from bottom to top and the backward computation from top to bottom.

* **Blue area — Inputs**: The input variables and their values. Each green node shows the value $v(\dots)$ used in the computation and, after backpropagation, the resulting $\text{grad}(\dots)$.
* **Yellow area — Operations**: The individual operations into which the expression has been decomposed. For an expression such as `(x**x) * y`, one operation computes $x^x$, and another multiplies that intermediate result by $y$. Each green operation node shows the expression and its value $v(\dots)$.
* **Pink trapezoids — Local derivatives**: These show how an operation changes with respect to one of its arguments. For example, for the multiplication $b \cdot y$, the local derivatives are $\frac{\partial (b \cdot y)}{\partial b} = y$ and $\frac{\partial (b \cdot y)}{\partial y} = b$. For $b = x^x$, the derivative with respect to $x$ is $x^x \cdot (\ln(x) + 1)$.
* **Green solid arrows — Forward computation ($f$)**: Values flow from the inputs through the operations toward the output. An edge labeled $f = \dots$ shows the numerical value being passed to the next operation.
* **Purple dotted arrows — Backward computation ($g$)**: Gradients flow in the opposite direction, from the output back toward the inputs. An edge labeled $g = \dots$ shows the gradient propagated along that path.
* **Green area — Output**: The final value of the complete expression. The backward computation starts at the output with $g = 1$.

During the forward pass, each operation receives values from below, computes its result, and passes that result upward. During the backward pass, each operation receives a gradient from above and uses its local derivatives to determine the gradients that are propagated toward its arguments.

When an input influences the output through more than one path—as $x$ does in $x^x$—the relevant derivative contributions are combined to obtain the final $\text{grad}(x)$.

*In short: green nodes and solid arrows represent values and forward computation; pink nodes and purple dotted arrows represent local derivatives and gradient propagation.*

In [1]:
import base64
import sympy as sp
from IPython.display import Image, display
from autodiff_visualizer import ComputationGraph

def render_mermaid(mermaid_code):
    # Encode the diagram code to base64
    graphbytes = mermaid_code.encode("utf8")
    base64_bytes = base64.b64encode(graphbytes)
    base64_string = base64_bytes.decode("ascii")
    # Display as an image using the mermaid.ink API
    display(Image(url="https://mermaid.ink/img/" + base64_string))


## 1. Introductory Example: $o = x \cdot y$

We start with the most basic building block to establish our visual language. Notice how the local derivative rules (e.g. `diff(x*y, x) = y`) are explicitly rendered between the inputs and the operation.

In [2]:
x, y = sp.symbols('x y')
expr1 = x * y

cg1 = ComputationGraph()
cg1.build(expr1, root_name="a")
cg1.evaluate({x: 2.0, y: 3.0})

render_mermaid(cg1.to_mermaid(output_name="o"))

## 2. A Simple Composite Graph: $o = (x \cdot x) + y$

Next, we introduce multiple operations and how gradients correctly propagate backwards across a deeper Directed Acyclic Graph (DAG).

In [3]:
expr2 = (x * x) + y

cg2 = ComputationGraph()
cg2.build(expr2, root_name="o")
cg2.evaluate({x: 3.0, y: 4.0})

render_mermaid(cg2.to_mermaid(output_name="m"))

## 3. The Lecture Example: $z = (x \cdot x) \cdot y + (y + 2)$

Here we reproduce a multi-path evaluation graph from the lectures, where gradients along converging paths (like the two paths from $y$) elegantly sum up thanks to standard reverse-mode gradient rules.

In [4]:
expr3 = (x * x) * y + (y + 2)

cg3 = ComputationGraph()
cg3.build(expr3, root_name="z")
cg3.evaluate({x: 3.0, y: 4.0})

render_mermaid(cg3.to_mermaid(output_name="m"))

## 4. More Nested Expressions: $L = ((x_1 + x_2) \cdot x_3)^2$

As formulas become arbitrarily deep, the hierarchical design keeps the flow of values and derivatives coherent.

In [5]:
x1, x2, x3 = sp.symbols('x_1 x_2 x_3')
expr4 = ((x1 + x2) * x3)**2

cg4 = ComputationGraph()
cg4.build(expr4, root_name="b")
cg4.evaluate({x1: 1.0, x2: 2.0, x3: 3.0})

render_mermaid(cg4.to_mermaid(output_name="L", grad_symbol="grad(L)"))